# arrodeio-llm — SFT v2.2: Tucano 2 (3.7B) as the base

Same data and recipe as v2.1 (847 pairs: 50 hand-written + 797 topic-grounded real-stanza chunks), but
the base model changes from **Tucano-1b1-Instruct** (1.1B, Llama, trained from scratch) to
**Polygl0t/Tucano2-qwen-3.7B-Instruct** (3.7B, Qwen3 continued-pretrained on Portuguese). Goal: better
coherence and rhyme from a stronger base. One variable changed on purpose: the base model.

**Runtime -> Change runtime type -> T4 GPU.** Expect ~40-50 min. The adapter is saved every 40 steps, so if the
session drops you still have a recent one in `model/cordel-sft-lora-t2-3b7`. If you hit an out-of-memory error, set
`SFT_BATCH` to `1` and `SFT_ACCUM` to `4` in cell 4 and re-run.

(Fallback if 3.7B won't fit/finish: set `SFT_BASE` to `Polygl0t/Tucano2-qwen-1.5B-Instruct` — ~10 min.)

### 1. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

### 2. Code + libraries

In [ ]:
%cd /content
!rm -rf arrodeio-llm && git clone --depth 1 https://github.com/bsana1/arrodeio-llm.git
%cd /content/arrodeio-llm
!pip install -q -U transformers peft accelerate
!pip uninstall -q -y torchao

### 3. Data

Already in the repo (`data/sft.jsonl` + `data/sft_v2.jsonl`, topics written by hand). Quick sanity check:

In [ ]:
!wc -l data/sft.jsonl data/sft_v2.jsonl
!head -c 700 data/sft_v2.jsonl

### 4. Train

`SFT_BATCH=2` x `SFT_ACCUM=2` keeps the effective batch at 4 like v2.1; gradient checkpointing (`SFT_GC=1`) makes it fit.

In [ ]:
import os
os.environ["SFT_BASE"] = "Polygl0t/Tucano2-qwen-3.7B-Instruct"
os.environ["SFT_DATA"] = "data/sft.jsonl,data/sft_v2.jsonl"
os.environ["SFT_OUT"] = "/content/arrodeio-llm/model/cordel-sft-lora-t2-3b7"
os.environ["SFT_MAXLEN"] = "512"
os.environ["SFT_BATCH"] = "2"
os.environ["SFT_ACCUM"] = "2"
os.environ["SFT_GC"] = "1"
!python scripts/sft.py

### 5. Download the adapter

In [ ]:
import shutil
shutil.make_archive('/content/cordel-sft-lora-t2-3b7', 'zip', 'model/cordel-sft-lora-t2-3b7')
from google.colab import files
files.download('/content/cordel-sft-lora-t2-3b7.zip')

### 6. Back on your laptop

```bash
cd ~/src/arrodeio-llm
unzip ~/Downloads/cordel-sft-lora-t2-3b7.zip -d model/cordel-sft-lora-t2-3b7
```
Then tell Claude — it compares against v2.1 on the same prompts before anything ships.